# News Recommender Model Selection (3 models) + Precision@K

Implements and compares:
1) LightFM (hybrid CF + content features)
2) implicit ALS (collaborative filtering baseline)
3) Sentence-Embedding ranker (MiniLM) using cosine similarity

Evaluation: Precision@K for K = 5, 10, 20

Notes:
- No DB connection (model selection only)
- Works with NewsAPI-style article JSON + preference JSON
- ALS cell includes safe ID remapping to avoid out-of-bounds errors


In [1]:
# Optional installs (uncomment if needed)
# !pip -q install lightfm implicit sentence-transformers scipy scikit-learn
#!pip install pyarrow==12.0.1

import numpy as np
import pandas as pd
from typing import Dict, List
from collections import defaultdict

RNG = np.random.default_rng(42)


In [2]:
seed_article = {
    "source": {"id": "techcrunch", "name": "TechCrunch"},
    "author": "Lauren Forristal",
    "title": "Bye-bye bots: Altera's game-playing AI agents get backing from Eric Schmidt | TechCrunch",
    "description": "Autonomous, AI-based players are coming to a gaming experience near you...",
    "url": "https://techcrunch.com/2024/05/08/bye-bye-bots-alteras-game-playing-ai-agents-get-backing-from-eric-schmidt/",
    "urlToImage": "https://techcrunch.com/wp-content/uploads/2024/05/Minecraft-keyart.jpg?resize=1200,720",
    "publishedAt": "2024-05-08T15:14:57Z",
    "content": "Autonomous, AI-based players are coming to a gaming experience near you..."
}

seed_user_pref = {
    "preferences": {
        "sources": ["reuters", "straits_times", "bbc"],
        "topics": ["finance", "tech", "sports"],
        "style": ["breaking", "lifestyle"]
    }
}

seed_article, seed_user_pref


({'source': {'id': 'techcrunch', 'name': 'TechCrunch'},
  'author': 'Lauren Forristal',
  'title': "Bye-bye bots: Altera's game-playing AI agents get backing from Eric Schmidt | TechCrunch",
  'description': 'Autonomous, AI-based players are coming to a gaming experience near you...',
  'url': 'https://techcrunch.com/2024/05/08/bye-bye-bots-alteras-game-playing-ai-agents-get-backing-from-eric-schmidt/',
  'urlToImage': 'https://techcrunch.com/wp-content/uploads/2024/05/Minecraft-keyart.jpg?resize=1200,720',
  'publishedAt': '2024-05-08T15:14:57Z',
  'content': 'Autonomous, AI-based players are coming to a gaming experience near you...'},
 {'preferences': {'sources': ['reuters', 'straits_times', 'bbc'],
   'topics': ['finance', 'tech', 'sports'],
   'style': ['breaking', 'lifestyle']}})

In [3]:
TOPICS = ["finance", "tech", "sports", "politics", "health", "entertainment", "science", "business"]
STYLES = ["breaking", "lifestyle", "analysis", "opinion", "how-to"]
SOURCES = [
    ("reuters", "Reuters"),
    ("bbc", "BBC"),
    ("straits_times", "The Straits Times"),
    ("techcrunch", "TechCrunch"),
    ("bloomberg", "Bloomberg"),
    ("espn", "ESPN"),
    ("theverge", "The Verge"),
    ("wsj", "Wall Street Journal"),
    ("cnbc", "CNBC"),
]

TOPIC_KEYWORDS = {
    "finance": ["stock", "market", "tariff", "rates", "inflation", "crypto", "earnings", "bank", "fund", "trading"],
    "tech": ["ai", "agent", "model", "chip", "gpu", "software", "startup", "app", "cloud", "cyber"],
    "sports": ["match", "league", "goal", "coach", "player", "tournament", "nba", "nfl", "soccer", "tennis"],
    "politics": ["election", "senate", "parliament", "minister", "policy", "government"],
    "health": ["vaccine", "hospital", "disease", "mental", "health", "clinic", "research"],
    "entertainment": ["movie", "music", "celebrity", "festival", "show", "series"],
    "science": ["study", "research", "space", "quantum", "physics", "biology"],
    "business": ["merger", "acquisition", "revenue", "company", "ceo", "growth", "funding"],
}

def infer_topics_from_text(text: str) -> List[str]:
    t = (text or "").lower()
    scored = []
    for topic, kws in TOPIC_KEYWORDS.items():
        score = sum(1 for kw in kws if kw in t)
        if score > 0:
            scored.append((topic, score))
    scored.sort(key=lambda x: -x[1])
    return [topic for topic, _ in scored[:2]] if scored else [str(RNG.choice(TOPICS))]

def make_article(i: int) -> Dict:
    sid, sname = SOURCES[int(RNG.integers(0, len(SOURCES)))]
    topic = str(RNG.choice(TOPICS, p=np.array([0.15, 0.18, 0.14, 0.10, 0.10, 0.10, 0.08, 0.15])))
    style = str(RNG.choice(STYLES, p=np.array([0.25, 0.20, 0.25, 0.15, 0.15])))

    title = f"{style.title()} update: {topic.title()} story #{i} from {sname}"
    desc = f"A {style} piece about {topic} with angles on {RNG.choice(['Asia','US','Europe','global'])}."
    content = " ".join([
        f"This {style} report covers {topic}.",
        " ".join(RNG.choice(TOPIC_KEYWORDS[topic], size=4, replace=True)),
        "Additional context and implications for readers."
    ])

    return {
        "source": {"id": sid, "name": sname},
        "author": f"Author {i}",
        "title": title,
        "description": desc,
        "url": f"https://example.com/{sid}/{i}",
        "urlToImage": None,
        "publishedAt": f"2024-05-{int(RNG.integers(1,29)):02d}T{int(RNG.integers(0,24)):02d}:{int(RNG.integers(0,60)):02d}:00Z",
        "content": content,
        "_topic": topic,
        "_style": style,
    }

articles = [seed_article] + [make_article(i) for i in range(1, 300)]
df_items = pd.DataFrame(articles)
df_items.head(3)


,source,author,title,description,url,urlToImage,publishedAt,content,_topic,_style
0,"{'id': 'techcrunch', 'name': 'TechCrunch'}",Lauren Forristal,Bye-bye bots: Altera's game-playing AI agents ...,"Autonomous, AI-based players are coming to a g...",https://techcrunch.com/2024/05/08/bye-bye-bots...,https://techcrunch.com/wp-content/uploads/2024...,2024-05-08T15:14:57Z,"Autonomous, AI-based players are coming to a g...",NaN,NaN
1,"{'id': 'reuters', 'name': 'Reuters'}",Author 1,How-To update: Sports story #1 from Reuters,A how-to piece about sports with angles on glo...,https://example.com/reuters/1,None,2024-05-15T23:44:00Z,This how-to report covers sports. match nba go...,sports,how-to
2,"{'id': 'theverge', 'name': 'The Verge'}",Author 2,Breaking update: Science story #2 from The Verge,A breaking piece about science with angles on ...,https://example.com/theverge/2,None,2024-05-26T18:38:00Z,This breaking report covers science. space qua...,science,breaking


In [4]:
def make_user_pref(u: int) -> Dict:
    sources = [SOURCES[i][0] for i in RNG.choice(len(SOURCES), size=3, replace=False)]
    topics = list(RNG.choice(TOPICS, size=3, replace=False))
    styles = list(RNG.choice(STYLES, size=2, replace=False))
    return {"preferences": {"sources": sources, "topics": topics, "style": styles}}

user_prefs = [seed_user_pref] + [make_user_pref(u) for u in range(1, 120)]
n_users = len(user_prefs)
n_items = len(articles)

def preference_match_score(article: Dict, pref: Dict) -> float:
    p = pref["preferences"]
    src = article["source"]["id"]
    topic = article.get("_topic") or infer_topics_from_text(
        (article.get("title","") + " " + article.get("description","") + " " + article.get("content",""))
    )[0]
    style = article.get("_style") or str(RNG.choice(STYLES))

    score = 0.0
    score += 1.0 if src in p["sources"] else 0.0
    score += 1.2 if topic in p["topics"] else 0.0
    score += 0.6 if style in p["style"] else 0.0
    score += float(RNG.normal(0, 0.15))
    return score

rows = []
for u, pref in enumerate(user_prefs):
    scores = np.array([preference_match_score(a, pref) for a in articles])
    m = int(RNG.integers(15, 40))
    top_idx = scores.argsort()[::-1][:120]
    chosen = RNG.choice(top_idx, size=m, replace=False)
    for i in chosen:
        rows.append((u, i, 1.0))

df_interactions = pd.DataFrame(rows, columns=["user_id", "item_id", "value"])
df_interactions.head(), df_interactions.shape


(   user_id  item_id  value
 0        0       70    1.0
 1        0      114    1.0
 2        0      232    1.0
 3        0      195    1.0
 4        0       20    1.0,
 (3224, 3))

In [5]:
from sklearn.model_selection import train_test_split

train_rows = []
test_rows = []

for u in range(n_users):
    user_items = df_interactions[df_interactions.user_id == u].item_id.values
    if len(user_items) < 5:
        continue
    tr, te = train_test_split(user_items, test_size=0.2, random_state=42)
    train_rows += [(u, i, 1.0) for i in tr]
    test_rows += [(u, i, 1.0) for i in te]

train = pd.DataFrame(train_rows, columns=["user_id","item_id","value"])
test  = pd.DataFrame(test_rows,  columns=["user_id","item_id","value"])

train.shape, test.shape


((2531, 3), (693, 3))

In [6]:
def precision_at_k(recommended: List[int], relevant_set: set, k: int) -> float:
    rec_k = recommended[:k]
    if k <= 0 or not rec_k:
        return 0.0
    hits = sum(1 for i in rec_k if i in relevant_set)
    return hits / k

def evaluate_model(reco_fn, users: List[int], k_list=(5,10,20)) -> pd.DataFrame:
    train_by_user = train.groupby("user_id")["item_id"].apply(set).to_dict()
    test_by_user  = test.groupby("user_id")["item_id"].apply(set).to_dict()

    out = []
    for u in users:
        seen = train_by_user.get(u, set())
        rel  = test_by_user.get(u, set())
        if not rel:
            continue
        ranked = reco_fn(u, seen)
        for k in k_list:
            out.append((u, k, precision_at_k(ranked, rel, k)))

    df = pd.DataFrame(out, columns=["user_id","k","precision"])
    return df.groupby("k", as_index=False)["precision"].mean()

users_eval = list(range(n_users))


In [7]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares

# ---- Remap IDs (contiguous) ----
train_als = train.copy()
test_als  = test.copy()

train_als["u"], user_uniques = pd.factorize(train_als["user_id"], sort=False)
train_als["i"], item_uniques = pd.factorize(train_als["item_id"], sort=False)

user_map = {uid: idx for idx, uid in enumerate(user_uniques)}
item_map = {iid: idx for idx, iid in enumerate(item_uniques)}
i2item = dict(enumerate(item_uniques))

test_als["u"] = test_als["user_id"].map(user_map)
test_als["i"] = test_als["item_id"].map(item_map)
test_als = test_als.dropna(subset=["u","i"]).copy()
test_als["u"] = test_als["u"].astype(int)
test_als["i"] = test_als["i"].astype(int)

n_users_als = int(train_als["u"].max()) + 1
n_items_als = int(train_als["i"].max()) + 1

print("ALS users:", n_users_als, "ALS items:", n_items_als)
print("Dropped test rows:", len(test) - len(test_als))

# ---- Build USER-ITEM matrix (rows=users, cols=items) ----
mat_als = sp.coo_matrix(
    (train_als["value"].values.astype(np.float32),
     (train_als["u"].values.astype(int), train_als["i"].values.astype(int))),
    shape=(n_users_als, n_items_als),
    dtype=np.float32
).tocsr()

# ---- Train ALS (NO TRANSPOSE) ----
alpha = 20.0
user_item = (mat_als * alpha).tocsr()   # rows=users, cols=items

als_model = AlternatingLeastSquares(
    factors=64,
    regularization=0.2,
    iterations=20,
    random_state=42
)
als_model.fit(user_item)

print("user_factors:", als_model.user_factors.shape, "item_factors:", als_model.item_factors.shape)

# ---- Correct sanity checks ----
assert als_model.user_factors.shape[0] == n_users_als, "user_factors count mismatch"
assert als_model.item_factors.shape[0] == n_items_als, "item_factors count mismatch"

print("✅ ALS trained correctly (users/items aligned).")


c:\Users\Sam\anaconda3\envs\newsrec\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Sam\anaconda3\envs\newsrec\lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: Intel MKL BLAS is configured to use 6 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


ALS users: 120 ALS items: 300
Dropped test rows: 0


100%|██████████| 20/20 [00:00<00:00, 1110.79it/s]

user_factors: (120, 64) item_factors: (300, 64)
✅ ALS trained correctly (users/items aligned).


In [8]:
def reco_als_manual(user_id: int, seen_train: set, N=200) -> list:
    u = user_map.get(user_id, None)
    if u is None:
        pop = train.groupby("item_id").size().sort_values(ascending=False).index.tolist()
        return [iid for iid in pop if iid not in seen_train]

    uvec = als_model.user_factors[u]
    scores = als_model.item_factors @ uvec  # (n_items,)

    # filter seen items
    seen_internal = [item_map[iid] for iid in seen_train if iid in item_map]
    if seen_internal:
        scores[np.array(seen_internal, dtype=int)] = -np.inf

    N_safe = min(N, n_items_als)
    top = np.argpartition(-scores, kth=min(N_safe-1, len(scores)-1))[:N_safe]
    top = top[np.argsort(-scores[top])]

    return [i2item[int(j)] for j in top]


In [9]:
# ==========================
# CELL: Test ALS recommendations + show top items for a user
# Requires:
# - als_model trained correctly (user_factors shape = (n_users_als, f), item_factors = (n_items_als, f))
# - user_map, item_map, i2item, mat_als, train, articles (list of dicts)
# - reco_als_manual(user_id, seen_train, N) defined
# ==========================

# 1) Quick sanity checks
print("user_factors:", als_model.user_factors.shape, "| expected users:", n_users_als)
print("item_factors:", als_model.item_factors.shape, "| expected items:", n_items_als)

assert als_model.user_factors.shape[0] == n_users_als, "ALS users mismatch (wrong matrix orientation)."
assert als_model.item_factors.shape[0] == n_items_als, "ALS items mismatch (wrong matrix orientation)."

# 2) Pick a user to inspect
u0 = 0  # change this to any user_id you want

# 3) Build seen set from train (original item_ids)
train_by_user = train.groupby("user_id")["item_id"].apply(set).to_dict()
seen0 = train_by_user.get(u0, set())

# 4) Get recommendations
topn = 10
recs = reco_als_manual(u0, seen0, N=topn)

print(f"\nALS top-{topn} recommendations for user_id={u0}")
print("Seen in train:", len(seen0))

# 5) Display nicely (uses your NewsAPI-like `articles` list)
rows = []
for rank, item_id in enumerate(recs, start=1):
    a = articles[item_id]
    rows.append({
        "rank": rank,
        "item_id": item_id,
        "source": a["source"]["id"],
        "topic": a.get("_topic", ""),
        "style": a.get("_style", ""),
        "title": a.get("title", "")[:120],
        "url": a.get("url", "")
    })

df_show = pd.DataFrame(rows)
display(df_show)

# 6) (Optional) Show whether these recommended items are in the test set (relevant)
test_by_user = test_als.groupby("user_id")["item_id"].apply(set).to_dict()
relevant = test_by_user.get(u0, set())

hits = [iid for iid in recs if iid in relevant]
print(f"\nHits in held-out test set for user {u0}: {len(hits)}/{topn}")
if hits:
    print("Hit item_ids:", hits)


user_factors: (120, 64) | expected users: 120
item_factors: (300, 64) | expected items: 300

ALS top-10 recommendations for user_id=0
Seen in train: 31


,rank,item_id,source,topic,style,title,url
0,1,280,straits_times,science,analysis,Analysis update: Science story #280 from The S...,https://example.com/straits_times/280
1,2,21,wsj,business,analysis,Analysis update: Business story #21 from Wall ...,https://example.com/wsj/21
2,3,192,straits_times,sports,breaking,Breaking update: Sports story #192 from The St...,https://example.com/straits_times/192
3,4,87,theverge,finance,breaking,Breaking update: Finance story #87 from The Verge,https://example.com/theverge/87
4,5,201,wsj,politics,opinion,Opinion update: Politics story #201 from Wall ...,https://example.com/wsj/201
5,6,205,cnbc,politics,lifestyle,Lifestyle update: Politics story #205 from CNBC,https://example.com/cnbc/205
6,7,269,techcrunch,business,opinion,Opinion update: Business story #269 from TechC...,https://example.com/techcrunch/269
7,8,17,straits_times,entertainment,lifestyle,Lifestyle update: Entertainment story #17 from...,https://example.com/straits_times/17
8,9,210,theverge,tech,analysis,Analysis update: Tech story #210 from The Verge,https://example.com/theverge/210
9,10,104,techcrunch,finance,analysis,Analysis update: Finance story #104 from TechC...,https://example.com/techcrunch/104



Hits in held-out test set for user 0: 0/10


In [10]:
embed_available = False
model_st = None

try:
    from sentence_transformers import SentenceTransformer
    model_st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embed_available = True
except Exception as e:
    print("sentence-transformers not available:", repr(e))
    from sklearn.feature_extraction.text import TfidfVectorizer

item_texts = []
for a in articles:
    t = " ".join([
        a.get("title") or "",
        a.get("description") or "",
        a.get("content") or "",
        f"source:{a['source']['id']}",
        f"topic:{a.get('_topic','')}",
        f"style:{a.get('_style','')}",
    ])
    item_texts.append(t)

user_texts = []
for pref in user_prefs:
    p = pref["preferences"]
    user_texts.append(" ".join([" ".join(p["sources"]), " ".join(p["topics"]), " ".join(p["style"])]))

if embed_available:
    item_emb = model_st.encode(item_texts, normalize_embeddings=True, show_progress_bar=False)
    user_emb = model_st.encode(user_texts, normalize_embeddings=True, show_progress_bar=False)
    print("✅ MiniLM embeddings:", item_emb.shape, user_emb.shape)
else:
    tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
    item_emb = tfidf.fit_transform(item_texts)
    user_emb = tfidf.transform(user_texts)
    print("⚠️ TF-IDF fallback:", item_emb.shape, user_emb.shape)


✅ MiniLM embeddings: (300, 384) (120, 384)


In [11]:
def reco_embed(user_id: int, seen_train: set) -> List[int]:
    if embed_available:
        sims = item_emb @ user_emb[user_id]  # cosine (normalized)
        ranked = np.argsort(-sims)
        return [int(i) for i in ranked if int(i) not in seen_train]
    else:
        from sklearn.metrics.pairwise import linear_kernel
        sims = linear_kernel(user_emb[user_id], item_emb).ravel()
        ranked = np.argsort(-sims)
        return [int(i) for i in ranked if int(i) not in seen_train]
print("ALS Precision@K")

emb_metrics = evaluate_model(reco_embed, users_eval, k_list=(5,10,20))
emb_metrics


ALS Precision@K


,k,precision
0,5,0.050000
1,10,0.053333
2,20,0.052083


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import numpy as np
import pandas as pd

# --- Build item texts (same idea as your MiniLM block) ---
item_texts_tfidf = []
for a in articles:
    t = " ".join([
        a.get("title") or "",
        a.get("description") or "",
        a.get("content") or "",
        f"source:{a['source']['id']}",
        f"topic:{a.get('_topic','')}",
        f"style:{a.get('_style','')}",
    ])
    item_texts_tfidf.append(t)

# --- Build user texts from preferences ---
user_texts_tfidf = []
for pref in user_prefs:
    p = pref["preferences"]
    user_texts_tfidf.append(" ".join([
        " ".join(p["sources"]),
        " ".join(p["topics"]),
        " ".join(p["style"]),
    ]))

# --- Fit TF-IDF on items and transform both items + users into same space ---
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_items = tfidf.fit_transform(item_texts_tfidf)     # (n_items, vocab)
X_users = tfidf.transform(user_texts_tfidf)         # (n_users, vocab)

print("✅ TF-IDF matrices:", X_items.shape, X_users.shape)

def reco_tfidf(user_id: int, seen_train: set, N=200) -> list:
    """
    Rank items by cosine similarity between user's TF-IDF vector and item TF-IDF vectors.
    Returns item indices (same indexing as `articles` list).
    """
    # similarity scores: (n_items,)
    sims = linear_kernel(X_users[user_id], X_items).ravel()

    # filter seen items
    if seen_train:
        sims[list(seen_train)] = -np.inf

    N_safe = min(N, len(sims))
    top = np.argpartition(-sims, kth=min(N_safe-1, len(sims)-1))[:N_safe]
    top = top[np.argsort(-sims[top])]
    return [int(i) for i in top]


✅ TF-IDF matrices: (300, 1690) (120, 1690)


In [13]:
# --- Evaluate TF-IDF model ---
tfidf_metrics = evaluate_model(reco_tfidf, users_eval, k_list=(5,10,20))
print("TF-IDF Precision@K")
display(tfidf_metrics)

# --- Show top-10 recommendations for one user ---
u0 = 0
train_by_user = train.groupby("user_id")["item_id"].apply(set).to_dict()
seen0 = train_by_user.get(u0, set())

topn = 10
recs = reco_tfidf(u0, seen0, N=topn)

# Check hits in held-out test
test_by_user = test.groupby("user_id")["item_id"].apply(set).to_dict()
rel0 = test_by_user.get(u0, set())
hits = [iid for iid in recs if iid in rel0]

print(f"\nTF-IDF top-{topn} recommendations for user_id={u0}")
print("Seen in train:", len(seen0))
print(f"Hits in held-out test set: {len(hits)}/{topn}")
if hits:
    print("Hit item_ids:", hits)

# Display table
rows = []
for rank, item_id in enumerate(recs, start=1):
    a = articles[item_id]
    rows.append({
        "rank": rank,
        "item_id": item_id,
        "source": a["source"]["id"],
        "topic": a.get("_topic",""),
        "style": a.get("_style",""),
        "title": (a.get("title","") or "")[:120],
        "url": a.get("url","")
    })

display(pd.DataFrame(rows))


TF-IDF Precision@K


,k,precision
0,5,0.066667
1,10,0.066667
2,20,0.065833



TF-IDF top-10 recommendations for user_id=0
Seen in train: 31
Hits in held-out test set: 0/10


,rank,item_id,source,topic,style,title,url
0,1,142,reuters,finance,lifestyle,Lifestyle update: Finance story #142 from Reuters,https://example.com/reuters/142
1,2,137,reuters,sports,lifestyle,Lifestyle update: Sports story #137 from Reuters,https://example.com/reuters/137
2,3,146,bbc,sports,lifestyle,Lifestyle update: Sports story #146 from BBC,https://example.com/bbc/146
3,4,254,bbc,sports,lifestyle,Lifestyle update: Sports story #254 from BBC,https://example.com/bbc/254
4,5,272,bbc,tech,breaking,Breaking update: Tech story #272 from BBC,https://example.com/bbc/272
5,6,39,straits_times,sports,lifestyle,Lifestyle update: Sports story #39 from The St...,https://example.com/straits_times/39
6,7,74,straits_times,tech,lifestyle,Lifestyle update: Tech story #74 from The Stra...,https://example.com/straits_times/74
7,8,80,straits_times,sports,breaking,Breaking update: Sports story #80 from The Str...,https://example.com/straits_times/80
8,9,42,straits_times,tech,breaking,Breaking update: Tech story #42 from The Strai...,https://example.com/straits_times/42
9,10,192,straits_times,sports,breaking,Breaking update: Sports story #192 from The St...,https://example.com/straits_times/192


In [14]:
test_by_user = test.groupby("user_id")["item_id"].apply(list).to_dict()
len(test_by_user.get(0, [])), test_by_user.get(0, [])[:10]


(8, [6, 293, 20, 173, 86, 215, 285, 127])

In [15]:
u0 = 0
test_items_u0 = test[test.user_id == u0].item_id.tolist()
print("User 0 test items:", test_items_u0)

# Show their metadata
rows = []
for iid in test_items_u0[:15]:
    a = articles[iid]
    rows.append({
        "item_id": iid,
        "source": a["source"]["id"],
        "topic": a.get("_topic",""),
        "style": a.get("_style",""),
        "title": a.get("title","")[:80],
    })

pd.DataFrame(rows)


User 0 test items: [6, 293, 20, 173, 86, 215, 285, 127]


,item_id,source,topic,style,title
0,6,straits_times,sports,analysis,Analysis update: Sports story #6 from The Stra...
1,293,theverge,finance,analysis,Analysis update: Finance story #293 from The V...
2,20,techcrunch,finance,analysis,Analysis update: Finance story #20 from TechCr...
3,173,straits_times,business,analysis,Analysis update: Business story #173 from The ...
4,86,bbc,business,breaking,Breaking update: Business story #86 from BBC
5,215,theverge,finance,breaking,Breaking update: Finance story #215 from The V...
6,285,bbc,health,lifestyle,Lifestyle update: Health story #285 from BBC
7,127,bbc,finance,analysis,Analysis update: Finance story #127 from BBC
